[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Ejercicio guiado: diccionario de datos de `penguins`.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo I: *Datos y su calidad*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se integrará lo visto en el Módulo I construyendo el diccionario de datos de `penguins`: el tipo de cada variable, su `dtype` adecuado y los posibles sesgos de recolección.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Este curso no repasa la API método por método: esa explicación detallada está en los *notebooks* del curso [py311](https://github.com/PythonistaMX/py311). Aquí se hace un repaso breve y el tiempo se dedica a las **decisiones de análisis**: qué tipo de dato elegir, qué hacer con un faltante y qué consecuencias tiene cada decisión. Para esta sesión:

| Tema | *Notebooks* de py311 |
|---|---|
| *NumPy* | [02 Conceptos básicos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/02_conceptos_basicos_de_numpy.ipynb) · [03 Gestión de arreglos](https://github.com/PythonistaMX/py311/blob/main/03_gestion_de_arreglos_de_numpy.ipynb) · [04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) · [05 Operaciones básicas con arreglos](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| *pandas* | [09 Introducción a *pandas*](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb) · [10 Tipos de datos](https://github.com/PythonistaMX/py311/blob/main/10_tipos_de_datos_de_pandas.ipynb) · [11 Operaciones básicas con *dataframes*](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) · [12 Índices y multiíndices](https://github.com/PythonistaMX/py311/blob/main/12_indices_y_multiindices.ipynb) · [13 Datos categóricos](https://github.com/PythonistaMX/py311/blob/main/13_datos_categoricos.ipynb) |

> ⚠️ **Nota: py311 usa *pandas* 2.** Sus *notebooks* se escribieron para *pandas* 2.x y este curso usa *pandas* 3, así que algunos resultados se verán distintos. Por ejemplo, allá el texto aparece con `dtype` `object` (aquí es `str`), se explica el parámetro `inplace` y la asignación encadenada como si modificaran el original (con *pandas* 3 ya no lo hacen sobre columnas o selecciones) y puede aparecer `SettingWithCopyWarning`, que ya no existe. La sección 1.1 de `03_pandas_3_esencial.ipynb` explica estos cambios. Cuando haya diferencias, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

Se importarán las bibliotecas con sus alias convencionales, se ubicará la carpeta `data/` (que ya trae los CSV del curso, sin necesidad de internet) y se fijará la semilla `401` para que los resultados aleatorios sean reproducibles.

* La siguiente celda importará las bibliotecas y definirá `DATA` y `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda mostrará las versiones instaladas. El curso usa *pandas* 3.

In [2]:
{"NumPy": np.__version__, "pandas": pd.__version__, "seaborn": sns.__version__}

{'NumPy': '2.5.4', 'pandas': '3.0.6', 'seaborn': '0.13.2'}

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda leerá `penguins.csv`, el *dataset* presentado en `03_pandas_3_esencial.ipynb`.

In [3]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
pinguinos.shape

(344, 7)

---
## 1. Ejercicio guiado: diccionario de datos de `penguins` (10 min + práctica).

Un **diccionario de datos** documenta cada columna: qué es, de qué tipo es, en qué unidades está y qué valores puede tomar. Es el primer entregable de cualquier EDA serio y se usará en el proyecto integrador.

### 🟢 Ejercicio 1a · Construye el diccionario.

Crea un *dataframe* `diccionario` con una fila por columna de `pinguinos` y las columnas: `columna`, `tipo_estadistico`, `dtype_pandas`, `unidad`, `valores_observados` (del mínimo al máximo observados si es numérica; lista de categorías si es de texto) y `n_faltantes`.

*Pista:* `dtype_pandas`, `valores_observados` y `n_faltantes` pueden calcularse con código; `tipo_estadistico` y `unidad` son conocimiento del dominio y se escriben a mano.

**Nota:** lo **observado** no es lo mismo que lo **posible**. Que las masas vayan de 2,700 a 6,300 g dice qué pesaron estos pingüinos, no cuánto puede pesar un pingüino. Los límites de lo posible (por ejemplo, de 2,500 a 6,500 g) se escriben con conocimiento del dominio, y en `07_duplicados_e_inconsistencias.ipynb` se usarán para detectar valores imposibles.

**🎯 Objetivo:** producir el primer entregable de un EDA: una tabla que documente cada columna. **Qué se espera aprender:** a combinar información calculada (tipos, rangos, faltantes) con conocimiento del dominio (unidades, tipo estadístico).

* La siguiente celda muestra una solución posible.

In [4]:
tipo_estadistico = {
    "species": "categórica nominal",
    "island": "categórica nominal",
    "bill_length_mm": "numérica continua",
    "bill_depth_mm": "numérica continua",
    "flipper_length_mm": "numérica continua (registrada como entero)",
    "body_mass_g": "numérica continua (registrada como entero)",
    "sex": "categórica nominal (binaria)",
}
unidad = {"bill_length_mm": "mm", "bill_depth_mm": "mm", "flipper_length_mm": "mm", "body_mass_g": "g"}

def valores_observados(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return f"{serie.min():g} – {serie.max():g}"
    return ", ".join(sorted(serie.dropna().unique()))

diccionario = pd.DataFrame({
    "columna": pinguinos.columns,
    "tipo_estadistico": [tipo_estadistico[c] for c in pinguinos.columns],
    "dtype_pandas": [str(pinguinos[c].dtype) for c in pinguinos.columns],
    "unidad": [unidad.get(c, "—") for c in pinguinos.columns],
    "valores_observados": [valores_observados(pinguinos[c]) for c in pinguinos.columns],
    "n_faltantes": pinguinos.isna().sum().to_numpy(),
})
diccionario

,columna,tipo_estadistico,dtype_pandas,unidad,valores_observados,n_faltantes
0,species,categórica nominal,str,—,"Adelie, Chinstrap, Gentoo",0
1,island,categórica nominal,str,—,"Biscoe, Dream, Torgersen",0
2,bill_length_mm,numérica continua,float64,mm,32.1 – 59.6,2
3,bill_depth_mm,numérica continua,float64,mm,13.1 – 21.5,2
4,flipper_length_mm,numérica continua (registrada como entero),float64,mm,172 – 231,2
5,body_mass_g,numérica continua (registrada como entero),float64,g,2700 – 6300,2
6,sex,categórica nominal (binaria),str,—,"FEMALE, MALE",11


### 🟢 Ejercicio 1b · Convierte los `dtypes`.

Crea `pinguinos_tipados` donde `species`, `island` y `sex` sean `category`, y `flipper_length_mm` y `body_mass_g` sean `Int64` (entero que admite faltantes). Verifica con `info()` y compara la memoria usada contra el original.

**Nota:** `Int64` **no** convierte la aleta y la masa en variables discretas: siguen siendo **medidas continuas**, como se anotó en el diccionario del ejercicio 1a. Aquí se usa `Int64` solo porque el archivo las **registra** en milímetros y gramos enteros y porque admite faltantes; dejarlas en `float64` también es una decisión válida. El formato de guardado no cambia cómo se analizan: se siguen resumiendo con media, mediana y desviación estándar, como cualquier medida.

**🎯 Objetivo:** aplicar las decisiones de la tabla `clasificacion` (sección 1.4 de `04_tipos_de_variables.ipynb`). **Qué se espera aprender:** que el `dtype` documenta cómo se guarda cada variable y, de paso, reduce la memoria, pero no cambia su tipo estadístico.

* La siguiente celda muestra una solución posible.

In [5]:
pinguinos_tipados = pinguinos.astype({
    "species": "category",
    "island": "category",
    "sex": "category",
    "flipper_length_mm": "Int64",
    "body_mass_g": "Int64",
})
pinguinos_tipados.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   species            344 non-null    category
 1   island             344 non-null    category
 2   bill_length_mm     342 non-null    float64 
 3   bill_depth_mm      342 non-null    float64 
 4   flipper_length_mm  342 non-null    Int64   
 5   body_mass_g        342 non-null    Int64   
 6   sex                333 non-null    category
dtypes: Int64(2), category(3), float64(2)
memory usage: 12.7 KB


* La siguiente celda comparará la memoria total de ambos *dataframes* (en bytes).

In [6]:
{"original": int(pinguinos.memory_usage(deep=True).sum()),
 "tipado": int(pinguinos_tipados.memory_usage(deep=True).sum()) if pinguinos_tipados is not None else None}

{'original': 25465, 'tipado': 12978}

### 🟢 Ejercicio 1c · Pregunta de sesgo.

Los datos se tomaron en **solo tres islas** del archipiélago Palmer y en **tres temporadas de anidación** (2007 a 2009). **Nota:** la versión del CSV que se usa no incluye la columna del año; el dato viene de la documentación del *dataset*.

¿Qué sesgo podría tener esta muestra? ¿Para qué población son válidas las conclusiones y qué **no** se debería afirmar?

**🎯 Objetivo:** relacionar el diseño de recolección de `penguins` (sección 1.4 de `05_muestreo_y_sesgos_de_recoleccion.ipynb`) con los límites de las conclusiones. **Qué se espera aprender:** a escribir el párrafo de "datos y limitaciones" que llevará el informe del proyecto.

**✅ Respuesta sugerida:**

* **Sesgo de selección / cobertura espacial:** tres islas no representan a todas las colonias de cada especie. Además, las especies no están repartidas igual (Gentoo solo en Biscoe, Chinstrap solo en Dream), así que **especie e isla están confundidas**: una diferencia "entre islas" puede ser en realidad una diferencia entre especies, y viceversa.
* **Cobertura temporal:** tres temporadas consecutivas pueden coincidir con condiciones particulares (alimento, clima, hielo marino) y no capturan tendencias de largo plazo.
* **Momento del muestreo / medición:** se mide durante la anidación; la masa corporal cambia a lo largo del ciclo reproductivo.
* **Selección (si se generaliza más allá de los reproductores):** solo se miden adultos que llegaron a reproducirse. Si la población de interés son los reproductores, no es un sesgo, sino la definición de la población; si se quiere hablar de todos los pingüinos (jóvenes y adultos que no anidan), sí lo es.

Las conclusiones valen para **pingüinos reproductores de esas colonias en 2007 a 2009**. No se debería generalizar a "todos los Adelie de la Antártida" ni atribuir diferencias a la isla sin considerar la especie.

### 🔴 Reto 1 · Muestreo por conveniencia simulado.

Simula que un equipo solo pudo medir a los **100 pingüinos más fáciles de atrapar**, con probabilidad de captura mayor para los más ligeros (proporcional a `1 / body_mass_g**3`). En palabras: cuanto más ligero es el pingüino, **mucho** más fácil es atraparlo; por ejemplo, uno de 3,000 g tiene unas 8 veces más probabilidad de caer en la muestra que uno de 6,000 g. Con `rng.choice(..., p=..., replace=False)` toma esa muestra y compara la masa media y la proporción de especies con las del *dataset* completo. ¿De qué signo es el sesgo?

**🎯 Objetivo:** simular un mecanismo de selección no probabilístico con probabilidades desiguales. **Qué se espera aprender:** a anticipar la **dirección** de un sesgo a partir de cómo se elige la muestra.

* La siguiente celda muestra una solución posible.

In [7]:
base = pinguinos.dropna(subset=["body_mass_g"]).reset_index(drop=True)
peso = 1 / base["body_mass_g"].to_numpy() ** 3
indices = rng.choice(len(base), size=100, replace=False, p=peso / peso.sum())
muestra = base.iloc[indices]
pd.DataFrame({
    "completo": base["species"].value_counts(normalize=True),
    "conveniencia": muestra["species"].value_counts(normalize=True),
}).round(3), base["body_mass_g"].mean().round(0), muestra["body_mass_g"].mean().round(0)

(           completo  conveniencia
 species                          
 Adelie        0.442          0.52
 Gentoo        0.360          0.27
 Chinstrap     0.199          0.21,
 np.float64(4202.0),
 np.float64(3963.0))

**✅ Respuesta sugerida:**

El sesgo es **negativo**: la muestra por conveniencia subestima la masa media y además cambia la composición de especies (hay menos Gentoo).

---
## ✅ Cierre.

**Hoy vimos:**
* *NumPy* en repaso exprés y cuatro decisiones que cambian resultados: `ddof`, `NaN`, desbordamiento de enteros y comparación de flotantes.
* *pandas* 3: *Copy-on-Write* obligatorio, el tipo `str` respaldado por *PyArrow* y por qué el código de *pandas* 2 puede comportarse distinto.
* *pandas* en repaso exprés (`read_csv`, `info`, `describe`, `loc`/`iloc`, filtros, `assign`) y por qué hay que reportar el denominador cuando hay faltantes.
* Tipos de variables (nominales, dicotómicas, ordinales, discretas y continuas), escalas de Stevens y cómo elegir su `dtype` (`str`, `category`, `Int64`, fechas), con las consecuencias de elegir mal.
* Muestreo: población, marco, parámetro y estadístico; diseños probabilísticos y no probabilísticos, comparados por simulación.
* Sesgos de recolección: selección, supervivencia, no respuesta y medición.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `07_duplicados_e_inconsistencias.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>